In [2]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os
from scipy.signal import find_peaks

In [3]:
#Function Definitions
def partial_x_2d(f,dx):
    """Compute ∂f/∂x using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dx = np.zeros_like(f)
    
    # Central difference in x-direction (axis=0)
    df_dx[1:Nx-1, :] = (f[2:Nx, :] - f[0:Nx-2, :]) / (2 * dx)
    
    # Periodic boundaries (x-direction)
    df_dx[0, :] = (f[1, :] - f[-1, :]) / (2 * dx)
    df_dx[-1, :] = (f[0, :] - f[-2, :]) / (2 * dx)
    
    return df_dx

def partial_y_2d(f,dy):
    """Compute ∂f/∂y using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dy = np.zeros_like(f)
    
    # Central difference in y-direction (axis=1)
    df_dy[:, 1:Ny-1] = (f[:, 2:Ny] - f[:, 0:Ny-2]) / (2 * dy)
    
    # Periodic boundaries (y-direction)
    df_dy[:, 0] = (f[:, 1] - f[:, -1]) / (2 * dy)
    df_dy[:, -1] = (f[:, 0] - f[:, -2]) / (2 * dy)
    
    return df_dy

def partial_x_fft(f, dx):
    """
    Calculate ∂f/∂x using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dx (float): Grid spacing in the x-direction.

    Returns:
        df_dx (2D np.array): Partial derivative ∂f/∂x.
    """
    nx, ny = f.shape
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi  # Wavenumbers in x
    kx = kx[:, np.newaxis]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dx_hat = 1j * kx * f_hat
    df_dx = np.fft.ifft2(df_dx_hat).real

    return np.array(df_dx)

def partial_y_fft(f, dy):
    """
    Calculate ∂f/∂y using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dy (float): Grid spacing in the y-direction.

    Returns:
        df_dy (2D np.array): Partial derivative ∂f/∂y.
    """
    nx, ny = f.shape
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi  # Wavenumbers in y
    ky = ky[np.newaxis, :]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dy_hat = 1j * ky * f_hat
    df_dy = np.fft.ifft2(df_dy_hat).real

    return df_dy

def index_of_just_smaller(arr, value):
    arr = np.asarray(arr)
    mask = arr < value
    if not np.any(mask):
        return None  # No smaller value exists
    return np.argmax(np.where(mask, arr, -np.inf))

def compute_radial_spectrum(fld, dx, dy, dtype=float):
    nx, ny = fld.shape

    # Fourier wavenumbers
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K_perp = np.sqrt(KX**2 + KY**2)

    # Power spectrum density
    ff = np.abs(np.fft.ifft2(fld))**2   # use fft2, not ifft2
    P_flat = ff.ravel()
    K_flat = K_perp.ravel()

    # Radial bins
    dk = kx[1] - kx[0]
    k_bins = np.arange(0, np.max(K_flat) + dk, dk)
    nk = len(k_bins)
    P_k = np.zeros(nk, dtype=dtype)

    # Bin indices (vectorized)
    idx_float = K_flat / dk
    i1 = np.floor(idx_float).astype(int)
    i2 = i1 + 1
    w = idx_float - i1   # weight to upper bin

    # Accumulate into bins
    for i in range(len(K_flat)):
        if i1[i] >= 0 and i2[i] < nk:
            P_k[i1[i]] += P_flat[i] * (1 - w[i])
            P_k[i2[i]] += P_flat[i] * w[i]

    # Pack into recarray
    tags = ['kk', 'sp']
    sps = np.recarray((nk,), dtype=[(t, dtype) for t in tags])
    sps['kk'] = k_bins
    sps['sp'] = P_k / dk

    return sps

In [4]:
# List of simulation directories
base_root = "/home/UNN/w24021992/NAS/simulations_beta_scan"

sim_dirs = [

    # f"{base_root}/beta_p_0.0625/beta_e_0.25",
    # f"{base_root}/beta_p_0.0625/beta_e_1",
    # f"{base_root}/beta_p_0.0625/beta_e_4",

    # f"{base_root}/beta_p_0.25/beta_e_1",
    # f"{base_root}/beta_p_0.25/beta_e_4",
    # f"{base_root}/beta_p_0.25/beta_e_16",

    # f"{base_root}/beta_p_1/beta_e_0.0625",
    # f"{base_root}/beta_p_1/beta_e_0.25",
    f"{base_root}/beta_p_1/beta_e_1",
    # f"{base_root}/beta_p_1/beta_e_4",
    # f"{base_root}/beta_p_1/beta_e_16",

    # f"{base_root}/beta_p_4/beta_e_0.0625",
    # f"{base_root}/beta_p_4/beta_e_0.25",
    # f"{base_root}/beta_p_4/beta_e_1",
    # # f"{base_root}/beta_p_4/beta_e_4",
    # f"{base_root}/beta_p_4/beta_e_16",

    # f"{base_root}/beta_p_16/beta_e_0.0625",
    # f"{base_root}/beta_p_16/beta_e_0.25",
    # f"{base_root}/beta_p_16/beta_e_16",
]

dx = 0.0625
dy = 0.0625

In [4]:
from concurrent.futures import ThreadPoolExecutor
from functools import partial

def compute_Jmax_at_t(t, sim_dir, dx):
    print(f"  t = {t}", flush=True)
    File_Bx = f"{sim_dir}/Bx_ApJ_t{t}.h5"
    File_By = f"{sim_dir}/By_ApJ_t{t}.h5"
    File_Bz = f"{sim_dir}/Bz_ApJ_t{t}.h5"
    with h5py.File(File_Bx, 'r') as fBx, \
         h5py.File(File_By, 'r') as fBy, \
         h5py.File(File_Bz, 'r') as fBz:
        Bx = fBx['DS1'][:].T
        By = fBy['DS1'][:].T
        Bz = fBz['DS1'][:].T

    Jx =  partial_y_fft(Bz, dx)
    Jy = -partial_x_fft(Bz, dx)
    Jz =  partial_x_fft(By, dx) - partial_y_fft(Bx, dx)
    mod_J = np.sqrt(Jx**2 + Jy**2 + Jz**2)

    J_max = np.max(mod_J)
    iy, ix = np.unravel_index(np.argmax(mod_J), mod_J.shape)
    x_peak = ix * dx
    y_peak = iy * dx
    return J_max, x_peak, y_peak, ix, iy

times = np.arange(0, 301)
N_WORKERS = 64   # tune to your cores / I/O

for sim_dir in sim_dirs:
    print(f"\nProcessing {sim_dir}")

    worker = partial(compute_Jmax_at_t, sim_dir=sim_dir, dx=dx)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        Jmax = np.array(list(ex.map(worker, times)))

    # ---------- Peak finding ----------
    peaks, props = find_peaks(Jmax, prominence=0.02 * np.max(Jmax))
    if len(peaks) == 0:
        print("  No significant peak found — skipping plot")
        continue

    first_good_peak = peaks[0]
    t_peak = times[first_good_peak]
    J_peak = Jmax[first_good_peak]
    print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")

    # ---------- Plot ----------
    plots_dir = os.path.join(sim_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 5))
    plt.plot(times, Jmax, lw=2, label=r'$\max(|\mathbf{J}|)$')
    plt.axvline(t_peak, linestyle='--', linewidth=1.5, color='gray')
    plt.scatter(t_peak, J_peak, zorder=5, color='red',
                label=f'Peak: t={t_peak}')
    plt.xlabel("Time")
    plt.ylabel(r"$\max(|\mathbf{J}|)$")
    plt.title("Maximum current density vs time")
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(plots_dir, "Reconnection_Time.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"  Saved: {save_path}")


Processing /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
  t = 0
  t = 1
  t = 2
  t = 3
  t = 4
  t = 5
  t = 6
  t = 7
  t = 8
  t = 9
  t = 10
  t = 11
  t = 12
  t = 13
  t = 14
  t = 15
  t = 16
  t = 17
  t = 18
  t = 19
  t = 20
  t = 21
  t = 22
  t = 23
  t = 24
  t = 25
  t = 26
  t = 27
  t = 28
  t = 29
  t = 30
  t = 31
  t = 32
  t = 33
  t = 34
  t = 35
  t = 36
  t = 37
  t = 38
  t = 39
  t = 40
  t = 41
  t = 42
  t = 43
  t = 44
  t = 45
  t = 46
  t = 47
  t = 48
  t = 49
  t = 50
  t = 51
  t = 52
  t = 53
  t = 54
  t = 55
  t = 56
  t = 57
  t = 58
  t = 59
  t = 60
  t = 61
  t = 62
  t = 63
  t = 64
  t = 65
  t = 66
  t = 67
  t = 68
  t = 69
  t = 70
  t = 71
  t = 72
  t = 73
  t = 74
  t = 75
  t = 76
  t = 77
  t = 78
  t = 79  t = 80

  t = 81
  t = 82  t = 83

  t = 84
  t = 85
  t = 86
  t = 87
  t = 88
  t = 89
  t = 90
  t = 91
  t = 92
  t = 93
  t = 94
  t = 95
  t = 96
  t = 97
  t = 98
  t = 99
  t = 100
  t = 101
  t = 102
  t 

ValueError: `x` must be a 1-D array

In [ ]:

# peaks, props = find_peaks(
#     Jmax,
#     prominence=0.02 * np.max(Jmax)
# )

# if len(peaks) == 0:
#     print("  No significant peak found — skipping plot")


# first_good_peak = peaks[0]
# t_peak = times[first_good_peak]
# J_peak = Jmax[first_good_peak]

# print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")

# # ---------- Plot ----------
# plots_dir = os.path.join(sim_dir, "plots")
# os.makedirs(plots_dir, exist_ok=True)

# plt.figure(figsize=(7, 5))
# plt.plot(times, Jmax, lw=2, label=r'$\max(|\mathbf{J}|)$')

# plt.axvline(t_peak, linestyle='--', linewidth=1.5)
# plt.scatter(t_peak, J_peak, zorder=5)

# plt.text(
#     t_peak,
#     J_peak,
#     f'  Peak\n  t = {t_peak}\n  $|J|_{{max}}$ = {J_peak:.3e}',
#     verticalalignment='bottom',
#     horizontalalignment='left',
#     bbox=dict(boxstyle="round", alpha=0.25)
# )

# plt.xlabel("Time step")
# plt.ylabel(r"$\max(|\mathbf{J}|)$")
# plt.title(f"Maximum current density vs time\n{sim_dir.split('/')[-1]}")
# plt.grid(True)
# plt.tight_layout()

# save_path = os.path.join(plots_dir, "Reconnection_Time.png")
# plt.savefig(save_path, dpi=300)
# plt.close()

# print(f"  Saved: {save_path}")

In [8]:
results = list(ex.map(worker, times))
Jmax   = np.array([r[0] for r in results])
x_pos  = np.array([r[1] for r in results])
y_pos  = np.array([r[2] for r in results])
ix_arr = np.array([r[3] for r in results])
iy_arr = np.array([r[4] for r in results])

RuntimeError: cannot schedule new futures after shutdown

In [9]:
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    results = list(ex.map(worker, times))   # MUST be inside the with

# now safe to use `results` here
Jmax   = np.array([r[0] for r in results])
x_pos  = np.array([r[1] for r in results])
y_pos  = np.array([r[2] for r in results])
ix_arr = np.array([r[3] for r in results])
iy_arr = np.array([r[4] for r in results])

  t = 0
  t = 1
  t = 2
  t = 3
  t = 4
  t = 5
  t = 6
  t = 7
  t = 8
  t = 9
  t = 10
  t = 11
  t = 12
  t = 13
  t = 14
  t = 15
  t = 16
  t = 17
  t = 18
  t = 19
  t = 20
  t = 21
  t = 22
  t = 23
  t = 24
  t = 25
  t = 26
  t = 27
  t = 28
  t = 29
  t = 30
  t = 31
  t = 32
  t = 33
  t = 34
  t = 35
  t = 36
  t = 37
  t = 38
  t = 39
  t = 40  t = 41

  t = 42
  t = 43
  t = 44
  t = 45
  t = 46
  t = 47
  t = 48
  t = 49
  t = 50
  t = 51
  t = 52
  t = 53
  t = 54
  t = 55
  t = 56
  t = 57
  t = 58
  t = 59
  t = 60
  t = 61
  t = 62
  t = 63
  t = 64
  t = 65
  t = 66
  t = 67
  t = 68
  t = 69
  t = 70
  t = 71
  t = 72
  t = 73
  t = 74
  t = 75
  t = 76
  t = 77
  t = 78
  t = 79
  t = 80
  t = 81
  t = 82
  t = 83
  t = 84
  t = 85
  t = 86
  t = 87
  t = 88
  t = 89
  t = 90
  t = 91
  t = 92
  t = 93
  t = 94
  t = 95
  t = 96
  t = 97
  t = 98
  t = 99
  t = 100
  t = 101
  t = 102
  t = 103
  t = 104
  t = 105
  t = 106
  t = 107
  t = 108
  t = 109
  t = 110


In [10]:
print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")
print(f"    located at (x, y) = ({x_pos[first_good_peak]:.3f}, "
      f"{y_pos[first_good_peak]:.3f})")
print(f"    grid indices (ix, iy) = ({ix_arr[first_good_peak]}, "
      f"{iy_arr[first_good_peak]})")

  First significant peak: t = 40, Jmax = 5.519e+00
    located at (x, y) = (252.688, 221.875)
    grid indices (ix, iy) = (4043, 3550)


In [5]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from scipy.signal import find_peaks


def compute_Jmax_at_t(t, sim_dir, dx):
    File_Bx = f"{sim_dir}/Bx_ApJ_t{t}.h5"
    File_By = f"{sim_dir}/By_ApJ_t{t}.h5"
    File_Bz = f"{sim_dir}/Bz_ApJ_t{t}.h5"
    try:
        with h5py.File(File_Bx, 'r') as fBx, \
             h5py.File(File_By, 'r') as fBy, \
             h5py.File(File_Bz, 'r') as fBz:
            Bx = fBx['DS1'][:].T
            By = fBy['DS1'][:].T
            Bz = fBz['DS1'][:].T
    except OSError:                                  # missing / unreadable snapshot
        print(f"  t = {t}: missing, skipped", flush=True)
        return (np.nan, np.nan, np.nan, np.nan, np.nan)

    print(f"  t = {t}", flush=True)
    Jx =  partial_y_fft(Bz, dx)
    Jy = -partial_x_fft(Bz, dx)
    Jz =  partial_x_fft(By, dx) - partial_y_fft(Bx, dx)
    mod_J = np.sqrt(Jx**2 + Jy**2 + Jz**2)
    J_max = np.max(mod_J)
    iy, ix = np.unravel_index(np.argmax(mod_J), mod_J.shape)
    x_peak = ix * dx
    y_peak = iy * dx
    return J_max, x_peak, y_peak, ix, iy


times = np.arange(0, 301)
N_WORKERS = 64   # tune to your cores / I/O

for sim_dir in sim_dirs:
    print(f"\nProcessing {sim_dir}")
    worker = partial(compute_Jmax_at_t, sim_dir=sim_dir, dx=dx)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = np.array(list(ex.map(worker, times)), dtype=float)   # (N, 5), input order preserved

    Jmax = results[:, 0]                             # 1-D max|J| series (NaN where a file was missing)
    # results[:, 1], results[:, 2] = x_peak, y_peak ; results[:, 3], results[:, 4] = ix, iy  (kept, unused)

    finite = np.isfinite(Jmax)
    if not finite.any():
        print("  No readable snapshots — skipping")
        continue

    # ---------- Peak finding (on the available snapshots) ----------
    t_fin, J_fin = times[finite], Jmax[finite]
    peaks, props = find_peaks(J_fin, prominence=0.02 * np.nanmax(Jmax))
    if len(peaks) == 0:
        print("  No significant peak found — skipping plot")
        continue
    first_good_peak = peaks[0]
    t_peak = t_fin[first_good_peak]
    J_peak = J_fin[first_good_peak]
    print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")

    # ---------- Plot ----------
    plots_dir = os.path.join(sim_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    plt.figure(figsize=(7, 5))
    plt.plot(times, Jmax, lw=2, label=r'$\max(|\mathbf{J}|)$')       # missing t show as gaps
    plt.axvline(t_peak, linestyle='--', linewidth=1.5, color='gray')
    plt.scatter(t_peak, J_peak, zorder=5, color='red',
                label=f'Peak: t={t_peak}')
    plt.xlabel("Time")
    plt.ylabel(r"$\max(|\mathbf{J}|)$")
    plt.title("Maximum current density vs time")
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(plots_dir, "Reconnection_Time.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"  Saved: {save_path}")


Processing /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
  t = 6
  t = 62
  t = 37
  t = 31
  t = 18
  t = 44
  t = 27
  t = 61
  t = 9
  t = 24
  t = 52
  t = 16
  t = 19
  t = 35
  t = 40
  t = 39
  t = 50
  t = 14
  t = 32
  t = 63
  t = 4
  t = 26
  t = 28
  t = 13
  t = 43
  t = 2
  t = 30
  t = 54
  t = 12
  t = 25
  t = 1
  t = 3
  t = 58
  t = 41
  t = 5
  t = 60
  t = 33
  t = 34
  t = 56
  t = 21
  t = 53
  t = 10
  t = 15
  t = 59
  t = 20
  t = 45
  t = 23
  t = 36
  t = 29
  t = 48
  t = 47
  t = 51
  t = 0
  t = 55
  t = 8
  t = 46
  t = 57
  t = 22
  t = 7
  t = 11
  t = 49
  t = 67
  t = 65
  t = 17
  t = 68
  t = 42
  t = 38
  t = 66
  t = 72
  t = 64
  t = 70
  t = 69
  t = 71
  t = 73
  t = 75
  t = 74
  t = 76
  t = 77
  t = 81
  t = 78
  t = 79
  t = 80
  t = 82
  t = 84
  t = 83
  t = 87
  t = 85
  t = 89
  t = 86
  t = 90
  t = 88
  t = 91
  t = 93
  t = 92
  t = 94
  t = 97
  t = 100
  t = 95
  t = 101
  t = 96
  t = 99
  t = 98
  t = 112
  t 